# Etiquetado semántico con LLM — Tarea 1

Pipeline basado en la propuesta de Mateo: fragmentación → embeddings multilingües → búsqueda semántica (FAISS) → validación con LLM.

**Mejoras respecto al notebook original:**
- Corre 100% en Google Colab (sin Ollama local) usando **Groq API** (gratuita)
- Modelo de embeddings **multilingüe** (`paraphrase-multilingual-MiniLM-L12-v2`) en lugar del modelo solo en inglés
- Cubre las **8 etiquetas** del proyecto (INTRO, BACK, METH, RES, DISC, CONTR, LIM, CONC)
- **Early stopping**: para cuando se alcanza la cuota por etiqueta, no procesa los 20k docs
- Desacoplamiento de patrones: el LLM evalúa coherencia retórica, no coincidencias de palabras clave
- Produce salidas compatibles con el pipeline DVC del proyecto

**Modelo LLM recomendado:** `llama-3.1-8b-instant` via Groq (gratuito, ~800 tokens/seg)  
**Alternativa:** `gemini-1.5-flash` via Google AI Studio (también gratuito)

**Cómo obtener la API key de Groq (gratis):**
1. Ve a https://console.groq.com y crea una cuenta
2. En `API Keys` genera una nueva clave
3. Pégala en la celda de configuración o en los Secrets de Colab

## 1. Instalación

In [ ]:
!pip install -q faiss-cpu sentence-transformers groq pyarrow pandas tqdm

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración

**Edita solo esta celda.** Cambia `NOMBRE`, `ETIQUETAS_ASIGNADAS` y `GROQ_API_KEY`.

In [ ]:
# ── Identificación ────────────────────────────────────────────────────────────
NOMBRE              = "Camilo"        # igual que en CARGA_TXT_MAIA_PROJECT
ETIQUETAS_ASIGNADAS = ["LIM", "CONC"]

# Referencia de asignaciones:
#   Jesus  -> ["INTRO", "BACK"]
#   Camilo -> ["LIM",   "CONC"]
#   Mateo  -> ["METH",  "RES"]
#   Sergio -> ["DISC",  "CONTR"]

# ── API ───────────────────────────────────────────────────────────────────────
# Opción A: pega tu clave directamente (no hagas commit de este notebook con la clave)
GROQ_API_KEY = "gsk_REEMPLAZA_CON_TU_CLAVE"

# Opción B (recomendada en Colab): usa Secrets
# from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

GROQ_MODEL = "llama3-70b-8192"   # gratuito, mayor calidad — recomendado para etiquetado
# Alternativa más rápida (menor calidad): "llama-3.1-8b-instant"

# ── Rutas ─────────────────────────────────────────────────────────────────────
INPUT_PATH        = f"/content/drive/MyDrive/{NOMBRE.lower()}_corpus.parquet"
OUTPUT_TRAIN_PATH = f"/content/drive/MyDrive/{NOMBRE.lower()}_task1_train.parquet"
OUTPUT_VALID_PATH = f"/content/drive/MyDrive/{NOMBRE.lower()}_task1_validacion.csv"

# ── Cuotas y fragmentación ────────────────────────────────────────────────────
MAX_POR_ETIQUETA = 2000   # fragmentos de entrenamiento por etiqueta
N_VALIDACION     = 200    # fragmentos para revisión humana por etiqueta
MIN_PALABRAS     = 250
MAX_PALABRAS     = 1000
CHUNK_SIZE       = 800    # palabras por chunk inicial
CHUNK_OVERLAP    = 200
TOP_K_RETRIEVAL  = 15     # candidatos por query semántica
LLM_SCORE_MIN    = 6      # umbral mínimo de calidad (1-10)
SEED             = 42

print(f"Miembro:             {NOMBRE}")
print(f"Etiquetas asignadas: {ETIQUETAS_ASIGNADAS}")
print(f"Modelo LLM:          {GROQ_MODEL}")
print(f"Cuota por etiqueta:  {MAX_POR_ETIQUETA} train + {N_VALIDACION} validación")

## 4. Definición de etiquetas

Cada etiqueta tiene queries semánticas en español para recuperar candidatos via FAISS.
Estas queries **no son los patrones de entrenamiento** — el LLM evalúa coherencia retórica,
no coincidencias léxicas, lo que desacopla el proceso de selección del aprendizaje del modelo.

In [ ]:
LABEL_CONFIG = {
    "INTRO": {
        "descripcion": "Introducción: presenta el problema, motivación y objetivo del trabajo",
        "queries": [
            "este artículo presenta un estudio sobre",
            "el objetivo de esta investigación es",
            "en este trabajo se propone abordar el problema de",
            "la motivación principal de este estudio es",
            "este trabajo surge de la necesidad de",
        ]
    },
    "BACK": {
        "descripcion": "Marco teórico / trabajos relacionados: revisión de literatura previa",
        "queries": [
            "estudios previos han demostrado que",
            "trabajos relacionados en el área de",
            "según la literatura existente sobre",
            "el estado del arte en este campo indica",
            "investigaciones anteriores han explorado",
        ]
    },
    "METH": {
        "descripcion": "Metodología: diseño experimental, datos, procedimientos y métodos aplicados",
        "queries": [
            "en este estudio se utilizó un método",
            "los datos fueron recolectados y procesados mediante",
            "se aplicó un modelo para evaluar",
            "la metodología propuesta consiste en",
            "se realizó un experimento con el fin de",
        ]
    },
    "RES": {
        "descripcion": "Resultados: métricas, hallazgos empíricos y tablas de resultados",
        "queries": [
            "los resultados muestran que",
            "se obtuvo una mejora significativa en",
            "los experimentos indican que el modelo",
            "la evaluación demuestra un desempeño de",
            "los valores obtenidos fueron superiores a",
        ]
    },
    "DISC": {
        "descripcion": "Discusión: interpretación de resultados, implicaciones y comparación con trabajos previos",
        "queries": [
            "estos resultados sugieren que",
            "en comparación con trabajos anteriores",
            "las implicaciones de estos hallazgos son",
            "esto puede explicarse por el hecho de que",
            "a diferencia de lo reportado por otros autores",
        ]
    },
    "CONTR": {
        "descripcion": "Contribuciones: aportes novedosos y originales del trabajo",
        "queries": [
            "la principal contribución de este trabajo es",
            "este estudio aporta una nueva perspectiva sobre",
            "a diferencia de enfoques previos, este trabajo propone",
            "entre los aportes originales de esta investigación",
            "este artículo presenta por primera vez",
        ]
    },
    "LIM": {
        "descripcion": "Limitaciones y trabajo futuro: restricciones del estudio y líneas abiertas",
        "queries": [
            "una limitación de este estudio es",
            "como trabajo futuro se propone",
            "este enfoque tiene la restricción de",
            "entre las limitaciones identificadas se encuentra",
            "futuras investigaciones podrían explorar",
        ]
    },
    "CONC": {
        "descripcion": "Conclusiones: síntesis final de hallazgos y cierre del artículo",
        "queries": [
            "en conclusión, este trabajo ha demostrado que",
            "los hallazgos principales de esta investigación son",
            "a modo de conclusión, podemos afirmar que",
            "en resumen, los resultados obtenidos indican",
            "este estudio concluye que",
        ]
    },
}

# Filtrar solo las etiquetas asignadas a este miembro
LABEL_CONFIG_ACTIVO = {k: v for k, v in LABEL_CONFIG.items() if k in ETIQUETAS_ASIGNADAS}

print("Etiquetas activas:")
for label, cfg in LABEL_CONFIG_ACTIVO.items():
    print(f"  {label}: {cfg['descripcion']}")

## 5. Imports y clientes

In [ ]:
import json
import time
import re
import os
import numpy as np
import pandas as pd
import faiss
from groq import Groq
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

groq_client = Groq(api_key=GROQ_API_KEY)

# Modelo multilingüe — funciona bien en español científico
# Alternativa de mayor calidad (más lenta): "intfloat/multilingual-e5-base"
print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Listo.")

## 6. Funciones del pipeline

In [ ]:
# ── Fragmentación ─────────────────────────────────────────────────────────────

def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Divide el texto en chunks de palabras con solapamiento."""
    words = text.split()
    chunks = []
    start = 0
    step = chunk_size - overlap
    while start < len(words):
        chunk_words = words[start:start + chunk_size]
        chunks.append(" ".join(chunk_words))
        start += step
    return chunks


def word_count(text):
    return len(text.split())


def expand_chunk(chunks, idx, min_words=MIN_PALABRAS, max_words=MAX_PALABRAS):
    """Expande un chunk hacia sus vecinos hasta alcanzar min_words sin superar max_words."""
    current = chunks[idx]
    total = word_count(current)
    left, right = idx - 1, idx + 1

    while total < min_words and (left >= 0 or right < len(chunks)):
        if left >= 0:
            candidate = chunks[left]
            if total + word_count(candidate) <= max_words:
                current = candidate + " " + current
                total += word_count(candidate)
                left -= 1
            else:
                left = -1
        if total >= min_words:
            break
        if right < len(chunks):
            candidate = chunks[right]
            if total + word_count(candidate) <= max_words:
                current = current + " " + candidate
                total += word_count(candidate)
                right += 1
            else:
                right = len(chunks)
        if left < 0 and right >= len(chunks):
            break

    return current

In [ ]:
# ── Embeddings y FAISS ────────────────────────────────────────────────────────

def build_faiss_index(chunks):
    embeddings = embedding_model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # similitud coseno sobre vectores normalizados
    index.add(np.array(embeddings, dtype="float32"))
    return index


def semantic_search(index, query, top_k=TOP_K_RETRIEVAL):
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(np.array(q_emb, dtype="float32"), top_k)
    # Devuelve (índice, score) para poder filtrar por confianza semántica
    return list(zip(indices[0], distances[0]))

In [ ]:
# ── Clasificación con LLM (Groq) ──────────────────────────────────────────────

LABEL_DESCRIPTIONS = "\n".join(
    f"- {k}: {v['descripcion']}" for k, v in LABEL_CONFIG.items()
)

def classify_with_llm(text, candidate_label, retries=3):
    """
    Pide al LLM que evalúe si el fragmento corresponde a candidate_label.
    Retorna {"label": str, "score": int, "razon": str} o None si falla.

    El prompt está diseñado para evaluar coherencia retórica, no presencia
    de palabras clave — esto desacopla la selección del entrenamiento.
    """
    prompt = f"""Eres un experto en análisis del discurso científico en español.

Tu tarea es determinar si el siguiente fragmento de un artículo científico pertenece
a la sección \"{candidate_label}" según su función retórica.

Definición de las secciones posibles:
{LABEL_DESCRIPTIONS}

IMPORTANTE: evalúa la FUNCIÓN COMUNICATIVA del texto, no si contiene
palabras clave específicas. Un fragmento puede pertenecer a una sección
aunque no mencione explícitamente su nombre.

Fragmento a evaluar:
\"\"\"\n{text[:2500]}\n\"\"\"

Responde ÚNICAMENTE con JSON válido, sin texto adicional:
{{
  "label": "INTRO" | "BACK" | "METH" | "RES" | "DISC" | "CONTR" | "LIM" | "CONC" | "OTRO",
  "score": número entero entre 1 y 10 (10 = certeza absoluta),
  "razon": "una oración explicando la decisión"
}}"""

    for attempt in range(retries):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=150,
            )
            raw = response.choices[0].message.content.strip()
            # Extraer JSON aunque haya texto extra alrededor
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [LLM] intento {attempt+1} falló: {e} — reintentando en {wait}s")
            time.sleep(wait)
    return None

In [ ]:
# ── Pipeline principal ────────────────────────────────────────────────────────

def process_document(text, doc_id, contadores):
    """
    Procesa un documento y retorna fragmentos etiquetados.
    contadores: dict {label: int} con cuántos fragmentos se han recolectado.
    Si todas las etiquetas ya alcanzaron la cuota, retorna lista vacía.
    """
    etiquetas_pendientes = [
        label for label in ETIQUETAS_ASIGNADAS
        if contadores.get(label, 0) < (MAX_POR_ETIQUETA + N_VALIDACION)
    ]
    if not etiquetas_pendientes:
        return []

    chunks = split_into_chunks(text)
    if len(chunks) < 2:
        return []

    index = build_faiss_index(chunks)

    # Recuperar candidatos por etiqueta
    candidatos = {}  # chunk_idx -> mejor label candidato
    for label in etiquetas_pendientes:
        for query in LABEL_CONFIG[label]["queries"]:
            for idx, score in semantic_search(index, query):
                if idx not in candidatos or score > candidatos[idx]["sem_score"]:
                    candidatos[idx] = {"candidate_label": label, "sem_score": float(score)}

    # Expandir, deduplicar y validar con LLM
    resultados = []
    textos_vistos = set()

    for idx, meta in sorted(candidatos.items(), key=lambda x: -x[1]["sem_score"]):
        candidate_label = meta["candidate_label"]
        if contadores.get(candidate_label, 0) >= (MAX_POR_ETIQUETA + N_VALIDACION):
            continue

        texto = expand_chunk(chunks, idx)
        wc = word_count(texto)
        if wc < MIN_PALABRAS or wc > MAX_PALABRAS:
            continue
        if texto in textos_vistos:
            continue
        textos_vistos.add(texto)

        clasificacion = classify_with_llm(texto, candidate_label)
        if clasificacion is None:
            continue

        label_llm = clasificacion.get("label", "OTRO")
        score_llm = clasificacion.get("score", 0)

        if label_llm in ETIQUETAS_ASIGNADAS and score_llm >= LLM_SCORE_MIN:
            contadores[label_llm] = contadores.get(label_llm, 0) + 1
            resultados.append({
                "doc_id":          doc_id,
                "texto":           texto,
                "etiqueta_auto":   label_llm,
                "llm_score":       score_llm,
                "llm_razon":       clasificacion.get("razon", ""),
                "sem_score":       meta["sem_score"],
                "num_palabras":    wc,
                "asignado_a":      NOMBRE,
            })

        # Pequeña pausa para respetar el rate limit de Groq (free: ~30 req/min)
        time.sleep(0.5)

    return resultados

## 7. Cargar corpus

In [ ]:
df_corpus = pd.read_parquet(INPUT_PATH)
# Mezclar para no sesgar hacia los primeros documentos del corpus
df_corpus = df_corpus.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Documentos disponibles: {len(df_corpus):,}")
print(f"Columnas: {df_corpus.columns.tolist()}")
df_corpus.head(2)

## 8. Ejecutar pipeline con early stopping

El loop para automáticamente cuando cada etiqueta alcanza su cuota
(`MAX_POR_ETIQUETA + N_VALIDACION`). No es necesario procesar los 20k documentos.

In [ ]:
todos_los_fragmentos = []
contadores = {label: 0 for label in ETIQUETAS_ASIGNADAS}
cuota_total = {label: MAX_POR_ETIQUETA + N_VALIDACION for label in ETIQUETAS_ASIGNADAS}

pbar = tqdm(df_corpus.iterrows(), total=len(df_corpus), desc="Documentos")

for _, fila in pbar:
    # Early stopping: todas las etiquetas cubiertas
    if all(contadores[l] >= cuota_total[l] for l in ETIQUETAS_ASIGNADAS):
        print("\nCuotas alcanzadas. Deteniendo.")
        break

    fragmentos = process_document(fila["texto"], fila["doc_id"], contadores)
    todos_los_fragmentos.extend(fragmentos)

    estado = " | ".join(f"{l}: {contadores[l]}/{cuota_total[l]}" for l in ETIQUETAS_ASIGNADAS)
    pbar.set_postfix_str(estado)

df_fragmentos = pd.DataFrame(todos_los_fragmentos)
print(f"\nFragmentos recolectados: {len(df_fragmentos):,}")
print(df_fragmentos["etiqueta_auto"].value_counts().to_string())

## 9. Separación train / validación

In [ ]:
# Validación: N_VALIDACION por etiqueta (para revisión humana con Kappa)
df_validacion = (
    df_fragmentos
    .groupby("etiqueta_auto", group_keys=False)
    .apply(lambda g: g.nlargest(min(N_VALIDACION, len(g)), "llm_score"))  # los de mayor confianza
    .reset_index(drop=True)
)

ids_val = set(df_validacion.index)

# Entrenamiento: resto, hasta MAX_POR_ETIQUETA por etiqueta
df_entrenamiento = (
    df_fragmentos[~df_fragmentos.index.isin(ids_val)]
    .groupby("etiqueta_auto", group_keys=False)
    .apply(lambda g: g.nlargest(min(MAX_POR_ETIQUETA, len(g)), "llm_score"))
    .reset_index(drop=True)
)

print(f"Entrenamiento: {len(df_entrenamiento):,} fragmentos")
print(df_entrenamiento["etiqueta_auto"].value_counts().to_string())
print()
print(f"Validación:    {len(df_validacion):,} fragmentos")
print(df_validacion["etiqueta_auto"].value_counts().to_string())

## 10. Guardar resultados

In [ ]:
# Entrenamiento como parquet
df_entrenamiento.to_parquet(OUTPUT_TRAIN_PATH, index=False)
mb_train = os.path.getsize(OUTPUT_TRAIN_PATH) / (1024 ** 2)

# Validación como CSV para anotación humana inter-anotador
planilla = pd.DataFrame({
    "doc_id":          df_validacion["doc_id"],
    "texto_corto":     df_validacion["texto"].str[:400],
    "etiqueta_auto":   df_validacion["etiqueta_auto"],
    "llm_score":       df_validacion["llm_score"],
    "llm_razon":       df_validacion["llm_razon"],
    "etiqueta_humano": "",   # ← rellenar en revisión
    "anotador":        NOMBRE,
    "notas":           "",
})
planilla.to_csv(OUTPUT_VALID_PATH, index=False)

print(f"Entrenamiento: {OUTPUT_TRAIN_PATH}  ({mb_train:.1f} MB, {len(df_entrenamiento):,} filas)")
print(f"Validación:    {OUTPUT_VALID_PATH}  ({len(planilla):,} filas)")
print()
print("Instrucciones de anotación:")
print("  1. Abre el CSV en Google Sheets")
print("  2. Lee 'texto_corto' y verifica si 'etiqueta_auto' es correcta")
print(f"  3. Escribe la etiqueta correcta en 'etiqueta_humano' (opciones: {ETIQUETAS_ASIGNADAS + ['OTRO']})")
print("  4. Comparte el CSV con al menos otro anotador para calcular Kappa")

## 11. Registro en DVC

Después de descargar el parquet de Drive, ejecuta esto en tu terminal local:

In [ ]:
nombre_lower = NOMBRE.lower()
destino = f"data/processed/task1/train/{nombre_lower}_task1_train.parquet"

print("Comandos para registrar en DVC:")
print()
print(f"  cp {OUTPUT_TRAIN_PATH} {destino}")
print(f"  dvc add {destino}")
print(f"  git add {destino}.dvc .gitignore")
print(f"  git commit -m 'data: add {nombre_lower} task1 training set (LLM-labeled)'")
print( "  git push")